# 📈🇵🇱 GPW Liga Radar — predykcja awansów i spadków w indeksach
## Część 1: Pozyskanie i przygotowanie danych

**Autor:** Wojciech Płonka (projekt indywidualny)

**Przedmiot:** Uczenie maszynowe w Python — laboratorium

---

Celem projektu jest klasyfikacja spółek warszawskiej giełdy do jednej z trzech „lig" indeksowych
(**WIG20**, **mWIG40**, **sWIG80**) na podstawie ich cech rynkowych. Spółki, które model przypisze do innej
ligi niż obecna, są kandydatami do awansu lub spadku przy najbliższej kwartalnej rewizji indeksów.

Ten notebook buduje zbiór danych: pobiera skład indeksów, mapuje spółki na symbole giełdowe, pobiera notowania
i wylicza cechy. Modele trenowane są w `02_modele.ipynb`.

## Konfiguracja środowiska

Dane rynkowe pobierane są biblioteką `yfinance` (notowania Yahoo Finance, spółki GPW mają sufiks `.WA`).
Skład indeksów odczytywany jest z Wikipedii, a mapowanie nazw spółek na symbole giełdowe — z wyszukiwarki
Yahoo Finance. W Google Colab `yfinance` instalujemy jednorazowo poleceniem `pip`.

In [ ]:
!pip -q install yfinance

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import requests, time, urllib.parse

sns.set_theme(style='whitegrid')
print('Środowisko gotowe ✅')

## Skład indeksów

Lista spółek wchodzących w skład każdego indeksu pobierana jest z odpowiedniej strony Wikipedii. Strona zawiera
wiele tabel (m.in. infobox), dlatego spośród nich wybierana jest ta, której liczba wierszy najlepiej odpowiada
znanej liczebności indeksu (20, 40, 80 spółek), a następnie wyodrębniana jest kolumna z nazwami spółek.

In [ ]:
INDEKSY = {
    'WIG20':  ('https://pl.wikipedia.org/wiki/WIG20', 20),
    'mWIG40': ('https://pl.wikipedia.org/wiki/MWIG40', 40),
    'sWIG80': ('https://pl.wikipedia.org/wiki/SWIG80', 80),
}

def wybierz_kolumne_z_nazwami(tabela):
    """Zwraca kolumnę zawierającą nazwy spółek (najwięcej unikalnych wartości tekstowych)."""
    najlepsza, max_tekstu = None, -1
    for kol in tabela.columns:
        wartosci = tabela[kol].astype(str)
        # ile wartości wygląda jak nazwa (litery, długość > 2, nie sama liczba)
        liczba = sum(w.strip()[:1].isalpha() and len(w.strip()) > 2 and not w.strip().replace('.','').replace(',','').isdigit() for w in wartosci)
        if liczba > max_tekstu:
            najlepsza, max_tekstu = kol, liczba
    return najlepsza

def pobierz_sklad(url, oczekiwana_liczba):
    tabele = pd.read_html(url)
    # wybierz tabelę o liczbie wierszy najbliższej oczekiwanej liczebności indeksu
    tabela = min(tabele, key=lambda t: abs(len(t) - oczekiwana_liczba))
    kol = wybierz_kolumne_z_nazwami(tabela)
    nazwy = (tabela[kol].astype(str)
             .str.replace(r'\[.*?\]', '', regex=True)   # przypisy [1]
             .str.split('(').str[0]                       # usuń nawiasy
             .str.strip())
    nazwy = [n for n in nazwy if n and n.lower() != 'nan' and len(n) > 2]
    return nazwy

sklad = {}
for liga, (url, n) in INDEKSY.items():
    nazwy = pobierz_sklad(url, n)
    sklad[liga] = nazwy
    print(f'{liga}: pobrano {len(nazwy)} spółek (oczekiwano ~{n})')
    print('   przykłady:', nazwy[:5])

## Mapowanie nazw na symbole giełdowe

Każda nazwa spółki tłumaczona jest na symbol notowań Yahoo Finance (z sufiksem `.WA`) przy użyciu wyszukiwarki
Yahoo. Dla nielicznych spółek, których wyszukiwarka nie rozpoznaje po nazwie, stosowany jest słownik ręcznych
poprawek. Mapowanie jest cache'owane, aby nie odpytywać tej samej nazwy wielokrotnie.

In [ ]:
# ręczne poprawki dla spółek trudnych do odnalezienia po nazwie
RECZNE = {
    'Comarch': 'CMR.WA', 'Asseco Poland': 'ACP.WA', 'Grupa Azoty': 'ATT.WA',
    'Bank Handlowy': 'BHW.WA', 'Bank Millennium': 'MIL.WA', 'Cyfrowy Polsat': 'CPS.WA',
    'Kęty': 'KTY.WA', 'Kety': 'KTY.WA', 'Develia': 'DVL.WA', 'Enea': 'ENA.WA',
}

_cache = {}
def nazwa_na_ticker(nazwa):
    if nazwa in _cache:
        return _cache[nazwa]
    for klucz, tk in RECZNE.items():
        if klucz.lower() in nazwa.lower():
            _cache[nazwa] = tk
            return tk
    try:
        url = 'https://query1.finance.yahoo.com/v1/finance/search?q=' + urllib.parse.quote(nazwa)
        r = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)
        wyniki = r.json().get('quotes', [])
        wa = [q['symbol'] for q in wyniki if q.get('symbol', '').endswith('.WA')]
        ticker = wa[0] if wa else None
    except Exception:
        ticker = None
    _cache[nazwa] = ticker
    time.sleep(0.2)   # nie zalewamy serwera zapytaniami
    return ticker

# budujemy tabelę: spółka -> ticker -> liga
wiersze = []
for liga, nazwy in sklad.items():
    for nazwa in nazwy:
        wiersze.append({'nazwa': nazwa, 'ticker': nazwa_na_ticker(nazwa), 'liga': liga})

spolki = pd.DataFrame(wiersze)
nieznalezione = spolki[spolki['ticker'].isna()]
print(f'Zmapowano {spolki["ticker"].notna().sum()} / {len(spolki)} spółek na tickery.')
if len(nieznalezione):
    print('Nie znaleziono tickera dla:', list(nieznalezione['nazwa']))
spolki = spolki.dropna(subset=['ticker']).drop_duplicates('ticker').reset_index(drop=True)
spolki.head()

## Pobranie notowań

Dla wszystkich spółek naraz pobierana jest roczna historia notowań (jedno zbiorcze zapytanie, co jest szybsze
i stabilniejsze niż pobieranie pojedynczo). Z notowań korzystać będziemy z cen zamknięcia oraz wolumenu —
to z nich wyliczymy cechy opisujące wielkość, płynność i zachowanie kursu.

In [ ]:
tickery = spolki['ticker'].tolist()
print(f'Pobieram roczne notowania dla {len(tickery)} spółek...')
notowania = yf.download(tickery, period='1y', interval='1d', auto_adjust=True, progress=False)
ceny = notowania['Close']
wolumen = notowania['Volume']
print('Pobrano. Kształt tabeli cen:', ceny.shape)

## Wyliczenie cech

Dla każdej spółki wyznaczane są cechy zbieżne z oficjalnymi kryteriami rankingu indeksów GPW (kapitalizacja
i wartość obrotów) oraz typowe miary zachowania kursu:

- **sredni_obrot** — średni dzienny obrót (cena × wolumen) w mln PLN; główna miara płynności i wielkości,
- **zmiennosc** — odchylenie standardowe dziennych stóp zwrotu (ryzyko),
- **momentum_3m / 6m / 12m** — stopa zwrotu w danym okresie,
- **cena_srednia** — przeciętny poziom kursu,
- **kapitalizacja** i **sektor** — pobierane dodatkowo z danych fundamentalnych (best-effort).

Kapitalizacja i sektor bywają niedostępne dla części mniejszych spółek, dlatego ich pobieranie jest zabezpieczone
i nie przerywa działania — braki uzupełnimy później.

In [ ]:
def stopa_zwrotu(seria, dni):
    s = seria.dropna()
    if len(s) < dni + 1:
        return np.nan
    return (s.iloc[-1] / s.iloc[-dni] - 1) * 100

cechy = []
for _, row in spolki.iterrows():
    tk = row['ticker']
    c = ceny[tk] if tk in ceny else pd.Series(dtype=float)
    v = wolumen[tk] if tk in wolumen else pd.Series(dtype=float)
    if c.dropna().empty:
        continue
    zwroty_dzienne = c.pct_change().dropna()
    cechy.append({
        'nazwa': row['nazwa'],
        'ticker': tk,
        'liga': row['liga'],
        'sredni_obrot_mln': float((c * v).mean()) / 1e6,
        'zmiennosc': float(zwroty_dzienne.std() * 100),
        'momentum_3m': stopa_zwrotu(c, 63),
        'momentum_6m': stopa_zwrotu(c, 126),
        'momentum_12m': stopa_zwrotu(c, 250),
        'cena_srednia': float(c.mean()),
    })

df = pd.DataFrame(cechy)
print('Spółki z policzonymi cechami:', len(df))
df.head()

In [ ]:
# kapitalizacja i sektor (best-effort, pojedynczo)
kap, sek = {}, {}
for tk in df['ticker']:
    try:
        info = yf.Ticker(tk).info
        kap[tk] = info.get('marketCap')
        sek[tk] = info.get('sector')
    except Exception:
        kap[tk], sek[tk] = None, None
    time.sleep(0.1)

df['kapitalizacja_mld'] = df['ticker'].map(kap).astype('float') / 1e9
df['sektor'] = df['ticker'].map(sek)
print('Kapitalizacja dostępna dla', df['kapitalizacja_mld'].notna().sum(), 'spółek')
df.head()

## Uzupełnienie braków

Braki w cechach liczbowych (np. krótka historia notowań, niedostępna kapitalizacja) uzupełniane są medianą
w obrębie danej ligi — wartość typowa dla spółek o podobnej wielkości jest lepszym przybliżeniem niż mediana
całego zbioru.

In [ ]:
kolumny_num = ['sredni_obrot_mln','zmiennosc','momentum_3m','momentum_6m','momentum_12m','cena_srednia','kapitalizacja_mld']
for kol in kolumny_num:
    df[kol] = df.groupby('liga')[kol].transform(lambda s: s.fillna(s.median()))
    df[kol] = df[kol].fillna(df[kol].median())   # gdyby cała liga była pusta
print('Braki po uzupełnieniu:', df[kolumny_num].isna().sum().sum())
df.describe()

## Eksploracja i wizualizacja

### Liczebność lig

W odróżnieniu od wielu zbiorów klasyfikacyjnych ten jest stosunkowo zbalansowany strukturalnie (20/40/80 spółek),
choć klasa sWIG80 jest najliczniejsza.

In [ ]:
plt.figure(figsize=(6,4))
ax = sns.countplot(x='liga', data=df, order=['WIG20','mWIG40','sWIG80'], palette='viridis')
plt.title('Liczba spółek w każdej lidze')
plt.xlabel(''); plt.ylabel('Liczba spółek')
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x()+p.get_width()/2, p.get_height()), ha='center', va='bottom')
plt.show()

### Obrót a liga

Średni obrót to kluczowa cecha — spodziewamy się, że spółki z WIG20 są wyraźnie bardziej płynne niż te z niższych
lig. Skala logarytmiczna pozwala porównać wartości różniące się o rzędy wielkości.

In [ ]:
plt.figure(figsize=(7,4))
sns.boxplot(x='liga', y='sredni_obrot_mln', data=df, order=['WIG20','mWIG40','sWIG80'], palette='viridis')
plt.yscale('log')
plt.title('Średni dzienny obrót w podziale na ligi')
plt.xlabel(''); plt.ylabel('Obrót [mln PLN, skala log]')
plt.show()

### Kapitalizacja a obrót

Wykres rozrzutu dwóch głównych kryteriów rankingu indeksów. Jeśli ligi tworzą czytelne skupiska, problem jest
dobrze rozdzielalny i klasyfikacja powinna działać; spółki na styku skupisk to potencjalni kandydaci do zmiany ligi.

In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(x='kapitalizacja_mld', y='sredni_obrot_mln', hue='liga',
                hue_order=['WIG20','mWIG40','sWIG80'], data=df, palette='viridis', s=70)
plt.xscale('log'); plt.yscale('log')
plt.title('Kapitalizacja vs obrót (skale log)')
plt.xlabel('Kapitalizacja [mld PLN]'); plt.ylabel('Średni obrót [mln PLN]')
plt.show()

### Korelacje cech

Sprawdzamy, które cechy są ze sobą powiązane (np. obrót i kapitalizacja) — silnie skorelowane cechy niosą zbliżoną
informację, co warto mieć na uwadze przy interpretacji modeli.

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df[kolumny_num].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Mapa ciepła korelacji cech')
plt.show()

## Zapis danych

Gotowy zbiór cech zapisywany jest do pliku CSV, z którego korzysta notebook z modelami.

In [ ]:
df.to_csv('gpw_cechy.csv', index=False)
print('Zapisano gpw_cechy.csv', df.shape)

---
## Podsumowanie

Zbudowano w pełni automatyczny zbiór danych o spółkach GPW: skład indeksów pobrano z Wikipedii, nazwy zmapowano na
symbole giełdowe, a z rocznych notowań wyliczono cechy opisujące wielkość, płynność i zachowanie kursu. Eksploracja
potwierdza, że obrót i kapitalizacja wyraźnie różnicują ligi, co zapowiada dobrą rozdzielalność problemu. Trening
i porównanie modeli klasyfikujących spółki do lig kontynuowane są w `02_modele.ipynb`.